[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Rate Limits


## What you will be able to do

Read how much of a rate limit is left from a response's headers, wait as long as a `429` asks,
whether its `Retry-After` is a number of seconds or a date, pace requests so that a limit is never
reached, back off with jitter when a refusal says nothing, and write a client that gives up instead
of waiting too long.


## The idea

### The problem

Every request so far was answered however soon it followed the last. A server that anyone can call
shares its time among everyone who calls it, and a client that sends a thousand requests a second
slows the API for all the others and costs its owner money. So most APIs limit how many requests a
client may send in a period, and answer a request past the limit with `429 Too Many Requests`
instead of the data.

That gives a client three jobs: notice the limit before it is refused, wait when it is refused, and
ask again without making things worse. A loop that asks again at once only collects more refusals,
and a thousand clients that all wait exactly one second come back together and are refused together.
The practice API's `/network/latest` allows 5 requests every 2 seconds, a limit small enough to reach
in a few lines.

### What a rate limit is

> A **rate limit** is the most requests a client may send an API in a **window** of time, such as 60
> an hour, counted for each key or each address. A response can say how much of the window is left,
> in headers such as `X-RateLimit-Remaining`, and when a new window starts. A request past the limit
> gets `429 Too Many Requests`, often with a **`Retry-After`** header that says how long to wait.
> **Backoff** is waiting longer after each refusal, and **jitter** is making each wait partly random,
> so that clients refused together do not come back together.

### Why it works that way

- **A limit shares a server fairly.** It keeps one busy client from slowing everyone else, and it
  keeps a bug in a loop from turning into a bill.
- **The headers tell, so a client need not guess.** A response that carries the limit, the requests
  left and the time until the window resets lets a client slow down before it is refused.
- **`Retry-After` is the server's own estimate.** It is a number of seconds or a date, and asking again
  sooner than it says only earns another `429`. A date is measured against the response's `Date`
  header, which comes from the same clock.
- **Waits double, and are partly random.** When a refusal does not say how long to wait, doubling the
  longest wait after every refusal backs away from a server that is struggling, and choosing each
  wait at random below that longest spreads out clients that were refused at the same moment.
- **A client gives up.** A limit that resets in an hour is not worth sitting through in a loop. After
  a few attempts, or before a wait longer than the program can spend, a client stops and says why.
- **The cheapest request is the one not sent.** Asking for the most a page holds, sending an `ETag`
  back for a `304`, and keeping what has not changed all leave more of a limit for the requests that
  matter.

### Where you will meet this

GitHub's API allows 60 requests an hour without authentication and 5,000 an hour for an
authenticated user. It reports the rest in `x-ratelimit-limit`, `x-ratelimit-remaining` and
`x-ratelimit-reset`, where the reset is a moment in UTC epoch seconds, not a number of seconds to
wait. It answers `GET /rate_limit` without counting it against that hourly limit, and its
documentation warns that an integration that keeps sending requests while limited may be banned.
Open-Meteo's free API allows fewer than 10,000 calls a day, 5,000 an hour and 600 a minute, and while
this guide was being built it answered some of ten requests sent at once with `429`, which is why
Setup in the notebooks that use it checks Open-Meteo four requests at a time. Nominatim,
OpenStreetMap's geocoder, allows at most one request a second. An IETF draft proposes standard
`RateLimit` and `RateLimit-Policy` headers.

### What this notebook covers

- A limited endpoint, and the headers that report its limit
- `/rate-limit`, which reports the limit without spending any of it
- A burst that runs out, and `429 Too Many Requests`
- Waiting as `Retry-After` asks, when it is a number of seconds and when it is a date
- Pausing when `X-RateLimit-Remaining` reaches `0`, so that no request is refused
- A throttle that spaces requests evenly
- Backoff with jitter, for a refusal that does not say how long to wait
- A client that stays inside a limit, and gives up rather than wait too long
- Five errors, from a burst sent all at once to backoff without jitter

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

for number in range(1, 7):
    response = requests.get("http://127.0.0.1:8765/network/latest", timeout=10)
    print(number, response.status_code, "remaining:", response.headers["X-RateLimit-Remaining"])
print("Retry-After:", response.headers["Retry-After"])
```

```
1 200 remaining: 4
2 200 remaining: 3
3 200 remaining: 2
4 200 remaining: 1
5 200 remaining: 0
6 429 remaining: 0
Retry-After: 2
```

Six requests in quick succession: five answered, each leaving one fewer, and a sixth refused, with
the number of seconds to wait before asking again.


## Setup

Nine imports, the last of them the practice API.

- `requests` sends every request, and a response's `headers` carries the limit
- `time` pauses with `sleep`, and measures with `monotonic`
- `random` chooses waits at random, from a seeded generator
- `email.utils` reads a `Retry-After` that is a date
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`

The practice API's limit runs on real time, so several examples take a few seconds, while they wait
as a client should.


In [1]:
import email.utils
import importlib
import random
import sys
import time
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### A limited endpoint, and the headers that report its limit

`/network/latest` sends the latest reading at every station that reports, and it is limited. Its
headers say how:


In [2]:
response = requests.get(f"{BASE}/network/latest", timeout=10)

print(response.status_code, response.json())
for name in ["X-RateLimit-Limit", "X-RateLimit-Remaining", "X-RateLimit-Reset"]:
    print(f"{name}: {response.headers[name]}")


200 [{'station': 'bergen', 'time': '2026-03-01T09:00Z', 'temperature_c': 2.7}, {'station': 'oslo', 'time': '2026-03-01T09:00Z', 'temperature_c': -4.2}, {'station': 'tromso', 'time': '2026-03-01T09:00Z', 'temperature_c': -6.3}]
X-RateLimit-Limit: 5
X-RateLimit-Remaining: 4
X-RateLimit-Reset: 2


The limit is 5 requests a window, this request used one, and the window closes in 2 seconds. It
opened with this request, the first since the last window closed. `X-RateLimit-Reset` is a number of
seconds here, where on GitHub's API the same header is a moment in time, which is one reason to read
the documentation of every API you call. Real limits are larger and their windows longer. The
practice API's are small so that a notebook reaches the limit in seconds, and like a limit on an
address, one limit covers everyone who calls it.

### /rate-limit: the limit, without spending it

A request to a limited endpoint spends some of the limit to find out how much is left. The practice
API answers `/rate-limit` without counting it, as GitHub's API does at `/rate_limit`. `wait_for_reset`
uses it to wait for the window to close, if any of it has been spent, so that every example below
starts with all five requests and prints the same thing on every run:


In [3]:
def wait_for_reset():
    """Wait until the limit's window closes, if any of its requests have been spent."""
    limit = requests.get(f"{BASE}/rate-limit", timeout=10).json()
    if limit["remaining"] < limit["limit"]:
        time.sleep(limit["reset"])


wait_for_reset()
print(requests.get(f"{BASE}/rate-limit", timeout=10).json())
requests.get(f"{BASE}/network/latest", timeout=10)
requests.get(f"{BASE}/network/latest", timeout=10)
print(requests.get(f"{BASE}/rate-limit", timeout=10).json())
print(requests.get(f"{BASE}/rate-limit", timeout=10).json())


{'limit': 5, 'remaining': 5, 'reset': 0}
{'limit': 5, 'remaining': 3, 'reset': 2}
{'limit': 5, 'remaining': 3, 'reset': 2}


The two requests to `/network/latest` spent two, and the requests to `/rate-limit` spent nothing.
`reset` is 0 while no window is open. `time.sleep` pauses the program for a number of seconds, and a
pause of `reset` seconds always reaches the end of the window, because the server rounds the seconds
left up to a whole number.

### A burst that runs out: 429 Too Many Requests

Seven requests, sent as fast as a loop can send them:


In [4]:
wait_for_reset()
statuses = []
for _ in range(7):
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    statuses.append(response.status_code)

print(statuses)
print(response.status_code, response.reason, response.json())
print("Retry-After:", response.headers["Retry-After"])


[200, 200, 200, 200, 200, 429, 429]
429 Too Many Requests {'error': 'rate limit exceeded: 5 requests every 2 seconds'}
Retry-After: 2


Five were answered and two refused. A refused request gets no data, and its `Retry-After` is the time
left in the window. The **Status Codes** notebook met `429` from `/status/429`, which always asks for
30 seconds. This one is a real limit, and the wait it asks for is real too.

### Waiting as the server asks: Retry-After

A client that meets a `429` waits as long as `Retry-After` asks, then sends the same request again.
`get_patiently` does that, and says when it waits. On its last attempt it stops waiting, and
`raise_for_status` raises the refusal as an error:


In [5]:
def get_patiently(url, attempts=3, **kwargs):
    """A response, sent again after each 429 once Retry-After's seconds have passed, at most `attempts` times."""
    for attempt in range(1, attempts + 1):
        response = requests.get(url, timeout=10, **kwargs)
        if response.status_code != 429 or attempt == attempts:
            response.raise_for_status()
            return response
        wait = int(response.headers["Retry-After"])
        print(f"  429 on attempt {attempt}: waiting {wait} seconds")
        time.sleep(wait)


wait_for_reset()
statuses = [get_patiently(f"{BASE}/network/latest").status_code for _ in range(8)]
print(statuses)


  429 on attempt 1: waiting 2 seconds
[200, 200, 200, 200, 200, 200, 200, 200]


The sixth request was refused, waited its 2 seconds, and was answered on its second attempt, in a new
window, which the last two requests used too. Nothing was lost, only delayed. `attempts` keeps a
server that never stops refusing from holding the program in the loop.

### A Retry-After that is a date

HTTP allows `Retry-After` to be a date instead of a number of seconds. `/beta/network/latest` is the
same endpoint as a future release will send it: the same limit, spending the same requests, with a
date in its `Retry-After`:


In [6]:
wait_for_reset()
for _ in range(6):
    response = requests.get(f"{BASE}/beta/network/latest", timeout=10)

print(response.status_code, "| Retry-After:", response.headers["Retry-After"], "| Date:", response.headers["Date"])


429 | Retry-After: Sun, 01 Mar 2026 09:00:02 GMT | Date: Sun, 01 Mar 2026 09:00:00 GMT


The date is the moment to ask again, by the server's clock. Subtracting the time on your own clock
from it would be wrong by however far the two clocks disagree, and the practice API's clock stopped
on March 1, 2026. Subtracting the response's own `Date`, from the same clock, gives the wait.
`retry_after_seconds` reads either form: `email.utils.parsedate_to_datetime` turns an HTTP date into
a `datetime`, and two `datetime`s subtract to a `timedelta`, whose `total_seconds` is the wait:


In [7]:
def retry_after_seconds(response):
    """The seconds a 429 asks a client to wait, from a Retry-After of seconds or of a date."""
    value = response.headers["Retry-After"]
    if value.isdigit():
        return int(value)
    retry_at = email.utils.parsedate_to_datetime(value)
    sent_at = email.utils.parsedate_to_datetime(response.headers["Date"])
    return max(0, (retry_at - sent_at).total_seconds())


print(retry_after_seconds(response), "seconds")
time.sleep(retry_after_seconds(response))
print(requests.get(f"{BASE}/beta/network/latest", timeout=10).status_code)


2.0 seconds
200


### Pausing before a refusal

A client that reads the limit headers need not be refused at all. This loop sends twelve requests,
and whenever `X-RateLimit-Remaining` reaches `"0"`, a string like every header value, it waits for
`X-RateLimit-Reset` seconds before sending the next:


In [8]:
wait_for_reset()
statuses = []
for _ in range(12):
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    statuses.append(response.status_code)
    if response.headers["X-RateLimit-Remaining"] == "0":
        time.sleep(int(response.headers["X-RateLimit-Reset"]))

print(statuses.count(200), "answered,", statuses.count(429), "refused")


12 answered, 0 refused


Every request was answered. After the fifth and the tenth, the loop waited out the window, so the
sixth and the eleventh each opened a new one. Waiting before a refusal takes as long as waiting after
one, but it spends no request on a `429`, and an API that sees a client keep meeting its limit may
stop answering that client: GitHub's documentation warns that an integration that keeps sending
requests while limited may be banned.

### Spacing requests evenly: a throttle

Some limits are stated only in documentation, with no headers to read, as Nominatim's one request a
second is. A throttle keeps under such a limit by spacing requests evenly, a little further apart
than the limit needs, because a request leaves the client a moment before the server counts it.
`time.monotonic` is a clock that only moves forward, which makes it the clock to measure waits with:


In [9]:
class Throttle:
    """Spaces calls so that no more than `limit` fall in any `window` seconds, with a margin to spare."""

    def __init__(self, limit, window, margin=0.05):
        self.interval = window / limit + margin
        self.last = None

    def wait(self):
        if self.last is not None:
            time.sleep(max(0, self.last + self.interval - time.monotonic()))
        self.last = time.monotonic()


wait_for_reset()
throttle = Throttle(limit=5, window=2)
start = time.monotonic()
statuses = []
for _ in range(8):
    throttle.wait()
    statuses.append(requests.get(f"{BASE}/network/latest", timeout=10).status_code)

print(statuses)
print("took at least 3 seconds:", time.monotonic() - start >= 3)


[200, 200, 200, 200, 200, 200, 200, 200]
took at least 3 seconds: True


The throttle spaced the requests 0.45 seconds apart, so no two-second window ever held more than five,
and eight requests took a little over three seconds. That is slower than the loop that paused at
`"0"`, which sent five at once and then waited, but a throttle never sends a burst, and an API can
limit bursts as well as totals, as Open-Meteo did with ten requests sent at once.

### Backoff with jitter, when a refusal says nothing

Not every `429` carries a `Retry-After`. Without one, a client backs off: the longest it may wait
doubles after every refusal, up to a cap, and the wait itself is chosen at random between 0 and that
longest. AWS's architecture blog compared ways of adding randomness to backoff, and found that this
one, which it calls full jitter, did the least work. `random.Random(11)` is a generator of random
numbers started from a seed, so that this notebook prints the same waits on every run. A real client
leaves the seed out:


In [10]:
def backoff(attempt, rng, cap=30):
    """Seconds to wait after refusal number attempt + 1: a random share of a longest that doubles each time."""
    return rng.uniform(0, min(cap, 2 ** attempt))


rng = random.Random(11)            # seeded, so that the waits are the same on every run
print("longest: ", [min(30, 2 ** attempt) for attempt in range(6)])
for client in ["a", "b", "c"]:
    print(f"client {client}:", [round(backoff(attempt, rng), 1) for attempt in range(6)])


longest:  [1, 2, 4, 8, 16, 30]
client a: [0.5, 1.1, 3.7, 3.7, 8.1, 17.6]
client b: [0.2, 1.0, 2.5, 6.3, 1.5, 9.1]
client c: [0.1, 1.6, 2.8, 0.3, 15.7, 28.9]


The longest wait doubles, 1, 2, 4, 8 and 16 seconds, and stops at the cap of 30. Every wait falls
somewhere below its longest, so three clients refused at the same moment send their next requests at
different moments, and the server meets them one at a time. A wait can come out short, as client c's
fourth did: what doubles is how long a wait may be. Common errors shows the same three clients
without the randomness.

### A client that stays inside the limit

The pieces of this notebook, in one client. `PoliteClient` names itself in `User-Agent`, as the
**Headers and Content Types** notebook advised, and never sends a request before the limit allows:
when a response leaves no requests in the window, or a request is refused, it sets the moment its
next request may go, from `X-RateLimit-Reset`, from `Retry-After` in either form, or from backoff
with jitter when there is no `Retry-After`. It tries at most `max_attempts` times, and raises an
error instead of starting a wait longer than `max_wait`:


In [11]:
class PoliteClient:
    """Requests to one API that stay inside its rate limit, and give up rather than wait too long."""

    def __init__(self, base, user_agent, max_attempts=4, max_wait=10, seed=None):
        self.base = base
        self.headers = {"User-Agent": user_agent}
        self.max_attempts = max_attempts
        self.max_wait = max_wait
        self.rng = random.Random(seed)
        self.resume_at = 0.0                   # a time.monotonic() reading: no request goes before it
        self.refused = 0
        self.pauses = 0

    def wait_until_allowed(self, path):
        seconds = self.resume_at - time.monotonic()
        if seconds > self.max_wait:
            raise RuntimeError(f"{path} asked for a wait of {seconds:.0f} seconds, more than {self.max_wait}")
        if seconds > 0:
            time.sleep(seconds)
            self.pauses += 1

    def get(self, path, **params):
        """The JSON at path, sent no sooner than the limit allows."""
        for attempt in range(1, self.max_attempts + 1):
            self.wait_until_allowed(path)
            response = requests.get(f"{self.base}{path}", params=params, headers=self.headers, timeout=10)
            if response.status_code == 429 and attempt < self.max_attempts:
                self.refused += 1
                if "Retry-After" in response.headers:
                    wait = retry_after_seconds(response)
                else:
                    wait = backoff(attempt - 1, self.rng, cap=self.max_wait)
                self.resume_at = time.monotonic() + wait
                continue
            response.raise_for_status()
            if response.headers.get("X-RateLimit-Remaining") == "0":
                self.resume_at = time.monotonic() + int(response.headers["X-RateLimit-Reset"])
            return response.json()


wait_for_reset()
client = PoliteClient(BASE, "station-report/1.0 (reports@example.com)")
for station in ["bergen", "oslo", "tromso"] * 4:
    client.get("/network/latest", station=station)
print("12 requests:", client.refused, "refused,", client.pauses, "pauses")

for _ in range(5):
    requests.get(f"{BASE}/network/latest", timeout=10)      # another part of the program spends the limit
print(client.get("/beta/network/latest", station="oslo"), "| refused:", client.refused)

try:
    client.get("/status/429")
except RuntimeError as error:
    print("gave up:", error)


12 requests: 0 refused, 2 pauses
{'station': 'oslo', 'time': '2026-03-01T09:00Z', 'temperature_c': -4.2} | refused: 1
gave up: /status/429 asked for a wait of 30 seconds, more than 10


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `resume_at`, set from `X-RateLimit-Reset` when `X-RateLimit-Remaining` is `"0"` | a pause before a refusal, read from headers that are text | Pausing before a refusal |
| `retry_after_seconds(response)` | `Retry-After` as seconds, or as a date measured against `Date` | A Retry-After that is a date |
| `backoff(attempt - 1, self.rng, cap=self.max_wait)` | a random wait under a longest that doubles | Backoff with jitter, when a refusal says nothing |
| `attempt < self.max_attempts`, then `raise_for_status()` | a client that stops trying, and says why | Waiting as the server asks: Retry-After |
| `time.monotonic()` | a clock that only moves forward, for measuring waits | Spacing requests evenly: a throttle |
| `seconds > self.max_wait` | a wait too long to sit through, refused before it starts | The idea, where a client gives up |

The twelve requests met no refusal, because the client paused twice. When another part of the
program spent the limit, the client's request to `/beta/network/latest` was refused, read the date in
its `Retry-After`, waited, and was answered. `/status/429` asked for 30 seconds, and a client that
should not stall for half a minute raised an error at once instead of sleeping through it. The client
sets its pause when a response arrives but waits only before its next request, so a program that
stops asking never waits at all.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/11-rate-limits-solutions.ipynb).

**1.** Send `/network/latest` one request for `station` set to `tromso`, and print the reading and the
three `X-RateLimit` headers.


In [12]:
# your code here


**2.** Call `wait_for_reset`, then print what `/rate-limit` reports before and after three requests to
`/network/latest`.


In [13]:
# your code here


**3.** Call `wait_for_reset`, then send requests to `/network/latest` until one is refused, and print
how many were answered and the refusal's `Retry-After`.


In [14]:
# your code here


**4.** Call `wait_for_reset`, then use `get_patiently` to fetch the latest reading at Oslo seven times,
and print the seven temperatures.


In [15]:
# your code here


**5.** Use `backoff` with `random.Random(7)` to print the waits, rounded to two decimal places, that a
client would make after its first five refusals, and their total.


In [16]:
# your code here


**6.** Call `wait_for_reset`, then use a `PoliteClient` with `max_wait` set to 1 to fetch the latest
reading at Bergen six times. Print how many readings arrived, and the error that stopped it.


In [17]:
# your code here


## Common errors

### HTTPError: 429 Client Error: Too Many Requests for url: http://127.0.0.1:8765/network/latest?station=tromso


In [18]:
wait_for_reset()
for station in ["bergen", "oslo", "tromso"] * 2:
    response = requests.get(f"{BASE}/network/latest", params={"station": station}, timeout=10)
    response.raise_for_status()


HTTPError: 429 Client Error: Too Many Requests for url: http://127.0.0.1:8765/network/latest?station=tromso

Six requests left as fast as the loop could send them, and the sixth went past the limit.
`raise_for_status` was right to raise, since the program did not get its reading, but nothing in the
request was wrong: it came too soon. Wait as `get_patiently` does, or pace the loop:


In [19]:
wait_for_reset()
temperatures = [get_patiently(f"{BASE}/network/latest", params={"station": station}).json()["temperature_c"]
                for station in ["bergen", "oslo", "tromso"] * 2]
print(temperatures)


  429 on attempt 1: waiting 2 seconds
[2.7, -4.2, -6.3, 2.7, -4.2, -6.3]


### No error, and 429 after 429: a retry sent without waiting


In [20]:
wait_for_reset()
for _ in range(5):
    requests.get(f"{BASE}/network/latest", timeout=10)

statuses = []
for attempt in range(4):
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    statuses.append(response.status_code)
    if response.status_code != 429:
        break

print(statuses)


[429, 429, 429, 429]


The loop tried again the moment it was refused, four times in a few milliseconds, and the window had
not closed for any of them. Every attempt was one more request for the server to refuse, from the
kind of client an API bans. Wait between attempts:


In [21]:
wait_for_reset()
for _ in range(5):
    requests.get(f"{BASE}/network/latest", timeout=10)

statuses = []
for attempt in range(4):
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    statuses.append(response.status_code)
    if response.status_code != 429:
        break
    time.sleep(retry_after_seconds(response))

print(statuses)


[429, 200]


### No error, and 429 within the limit: a loop that counted only its own requests


In [22]:
wait_for_reset()
requests.get(f"{BASE}/network/latest", timeout=10)      # a request from another part of the program
requests.get(f"{BASE}/network/latest", timeout=10)      # and another

statuses = []
for _ in range(5):                                       # five, the limit, so no 429 is expected
    statuses.append(requests.get(f"{BASE}/network/latest", timeout=10).status_code)

print(statuses)


[200, 200, 200, 429, 429]


The loop sent five requests, the limit, and two were refused. A limit counts every request made with
the same key or from the same address, and two requests elsewhere in the program had already spent
part of it, as another script or another notebook running at the same time would. A count kept by
one loop knows nothing of them. The server's count does, so read it from every response:


In [23]:
wait_for_reset()
requests.get(f"{BASE}/network/latest", timeout=10)
requests.get(f"{BASE}/network/latest", timeout=10)

statuses = []
for _ in range(5):
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    statuses.append(response.status_code)
    if response.headers["X-RateLimit-Remaining"] == "0":
        time.sleep(int(response.headers["X-RateLimit-Reset"]))

print(statuses)


[200, 200, 200, 200, 200]


### ValueError: invalid literal for int() with base 10: 'Sun, 01 Mar 2026 09:00:02 GMT'


In [24]:
wait_for_reset()
for _ in range(6):
    response = requests.get(f"{BASE}/beta/network/latest", timeout=10)

time.sleep(int(response.headers["Retry-After"]))


ValueError: invalid literal for int() with base 10: 'Sun, 01 Mar 2026 09:00:02 GMT'

`int` reads a number of seconds, and this `Retry-After` is a date, which HTTP allows. The code worked
against every endpoint that sent seconds, and failed on the first that did not. Read either form with
`retry_after_seconds`:


In [25]:
print(retry_after_seconds(response), "seconds")

time.sleep(retry_after_seconds(response))
print(requests.get(f"{BASE}/beta/network/latest", timeout=10).status_code)


2.0 seconds
200


### No error, and every client back at once: backoff without jitter


In [26]:
for client in ["a", "b", "c"]:
    moments, now = [], 0
    for attempt in range(4):
        now += min(30, 2 ** attempt)                     # the longest wait, every time
        moments.append(now)
    print(f"client {client} asks again at", moments)


client a asks again at [1, 3, 7, 15]
client b asks again at [1, 3, 7, 15]
client c asks again at [1, 3, 7, 15]


Waiting the longest wait every time doubles the waits, and three clients refused at the same moment
double the same waits, so all three ask again at 1, 3, 7 and 15 seconds. A server that refused them
for arriving together meets them together again, four times. Choosing each wait at random, as
`backoff` does, spreads them out:


In [27]:
rng = random.Random(11)            # seeded, so that the moments are the same on every run
for client in ["a", "b", "c"]:
    moments, now = [], 0
    for attempt in range(4):
        now += backoff(attempt, rng)
        moments.append(round(now, 2))
    print(f"client {client} asks again at", moments)


client a asks again at [0.45, 1.57, 5.27, 8.99]
client b asks again at [0.51, 1.68, 2.42, 6.52]
client c asks again at [0.63, 2.22, 2.59, 5.02]


## Recap

- A rate limit is the most requests a client may send in a window, counted for its key or address,
  and a request past it gets `429 Too Many Requests`.
- `X-RateLimit-Limit`, `X-RateLimit-Remaining` and `X-RateLimit-Reset` report the limit as text, and
  whether `Reset` means seconds or a moment depends on the API.
- After a `429`, wait as long as `Retry-After` asks. A date in it is measured against the response's
  `Date`, not your own clock.
- Pause when `X-RateLimit-Remaining` reaches `0`, or space requests with a throttle, and no request is
  refused.
- When a refusal gives no `Retry-After`, back off with jitter: a random wait below a longest that
  doubles.
- Stop after a few attempts, and before a wait longer than the program can spend.


## What is next

The **Errors and Retries** notebook. Every failure here was a limit, and the server said how long to
wait. That notebook meets failures that say less: a request that times out, a connection that drops,
and a server that fails, and decides which of them are worth trying again.


---

&#8592; **Previous:** [Pagination](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/10-pagination.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
